# 01 - Ingesta Bronze
Este notebook implementa la ingesta batch del dataset de operaciones BYMA hacia la capa bronze.

## Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Paths
repo_path = "/Workspace/Repos/francokohn/iol-byma-data-engineer-challenge"
file_path = f"{repo_path}/data/operaciones_byma_2026.csv"

dbfs_path = "dbfs:/FileStore/tables/operaciones_byma_2026.csv"

In [0]:
schema = StructType([
    StructField("fecha", StringType(), True),
    StructField("tipoTran", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("descripcion_titulo", StringType(), True),
    StructField("moneda", StringType(), True),
    StructField("simbolo_titulo", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DecimalType(18, 4), True),
    StructField("id_transaccion", StringType(), True),
    StructField("origen", StringType(), True)
])

In [0]:
df_raw = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("dbfs:/Volumes/workspace/default/iol_data/operaciones_byma_2026.csv")

In [0]:
df = df_raw \
    .withColumn("fecha_ts_utc", to_timestamp("fecha")) \
    .withColumn("fecha_ts_local", from_utc_timestamp("fecha_ts_utc", "America/Argentina/Buenos_Aires")) \
    .withColumn("fecha_particion", to_date("fecha_ts_local")) \
    .withColumn("_ingested_at", current_timestamp())

In [0]:
display(df.select("fecha", "fecha_ts_utc", "fecha_ts_local", "fecha_particion").limit(10))

In [0]:
df_duplicates = df.groupBy("id_transaccion") \
    .count() \
    .filter(col("count") > 1) \
    .withColumn("fecha_deteccion", current_timestamp())

In [0]:
display(df_duplicates)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_byma")

In [0]:
df_dedup = df.dropDuplicates(["id_transaccion"])

In [0]:
df_dedup.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("fecha_particion") \
    .saveAsTable("bronze_byma.operaciones_raw")

In [0]:
df_duplicates.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_byma.calidad_duplicados")

In [0]:
%sql 
SELECT COUNT(*) FROM bronze_byma.operaciones_raw

In [0]:
%sql 
SELECT COUNT(*) FROM bronze_byma.calidad_duplicados